In [ ]:
"""
Week 3 - Metric Calculation and Data Modeling
Issue #7: Calculate Total Spend, CPC, CAC, ROAS per channel per attribution model
Issue #8: Build a Star Schema (fact + dimension tables) for BI tools
"""
import sqlite3
import pandas as pd

DB_PATH = "data/attribution.db"
AVG_ORDER_VALUE = 120.0  # assumption, documented in README

ATTRIBUTION_VIEWS = {
    "first_touch": "channel_credit_first_touch",
    "last_touch": "channel_credit_last_touch",
    "linear": "channel_credit_linear",
}


def build_dim_channel(conn):
    spend = pd.read_sql(
        """
        SELECT CampaignChannel AS channel,
               SUM(AdSpend) AS total_spend,
               SUM(WebsiteVisits) AS total_clicks
        FROM customers
        GROUP BY CampaignChannel
        """,
        conn,
    )
    spend.to_sql("dim_channel_spend", conn, if_exists="replace", index=False)
    return spend


def build_fact_kpis(conn, spend_df):
    frames = []
    for model_name, view in ATTRIBUTION_VIEWS.items():
        credit = pd.read_sql(f"SELECT * FROM {view}", conn)
        merged = credit.merge(spend_df, on="channel", how="left")
        merged["attribution_model"] = model_name
        merged["cpc"] = merged["total_spend"] / merged["total_clicks"].replace(0, pd.NA)
        merged["cac"] = merged["total_spend"] / merged["attributed_conversions"].replace(0, pd.NA)
        merged["revenue"] = merged["attributed_conversions"] * AVG_ORDER_VALUE
        merged["roas"] = merged["revenue"] / merged["total_spend"].replace(0, pd.NA)
        frames.append(merged)

    fact = pd.concat(frames, ignore_index=True)
    fact.to_sql("fact_channel_kpis", conn, if_exists="replace", index=False)
    return fact


if __name__ == "__main__":
    conn = sqlite3.connect(DB_PATH)
    spend_df = build_dim_channel(conn)
    fact = build_fact_kpis(conn, spend_df)
    conn.commit()
    conn.close()
    pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
    print(fact[["attribution_model", "channel", "total_spend", "attributed_conversions", "cac", "roas"]]
          .sort_values(["attribution_model", "roas"], ascending=[True, False]))
